# NeuraSight — Chest X-Ray Stacking Ensemble

Trains a **Logistic Regression meta-learner** on softmax
probabilities from three base models (EfficientNet, ResNet,
DenseNet) with a leakage-safe evaluation split.

Requires `Chest_Xray_Model_Training.ipynb` to have run first.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install seaborn -q

In [ ]:
import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize

SEED = 42
np.random.seed(SEED)

## 2. Configuration

In [ ]:
MODEL_ORDER = ["efficientnet", "resnet", "densenet"]
SAVE_NAMES = {
    "efficientnet": "CHEST_XRAY_EFFICIENTNET",
    "resnet": "CHEST_XRAY_RESNET",
    "densenet": "CHEST_XRAY_DENSENET",
}
CLASS_NAMES = ["Normal", "Pneumonia", "Tuberculosis"]
NUM_CLASSES = 3
SAVE_DIR = "/content/drive/MyDrive/NeuraSight/models"
ENSEMBLE_DIR = "/content/drive/MyDrive/NeuraSight/ensemble/chest_xray"

os.makedirs(ENSEMBLE_DIR, exist_ok=True)
print('Models dir:', SAVE_DIR)
print('Ensemble dir:', ENSEMBLE_DIR)

## 3. Load Saved Probabilities

Load softmax probabilities and labels from training notebook.
Concatenate into feature matrix X (N, 9).

In [ ]:
# Load probabilities from each base model
probs = {}
for key in MODEL_ORDER:
    sn = SAVE_NAMES[key]
    path = os.path.join(SAVE_DIR, sn + '_test_probs.npy')
    probs[key] = np.load(path)
    print(f'{key}: shape {probs[key].shape}')

# Load ground truth labels
labels = np.load(os.path.join(SAVE_DIR, 'test_labels.npy'))
print(f'Labels: shape {labels.shape}')
print(f'Class distribution: {np.bincount(labels)}')

# Concatenate: (N, 3*3=9) features
X = np.hstack([probs[key] for key in MODEL_ORDER])
y = labels
print(f'\nFeature matrix X: {X.shape}')
print(f'Labels y: {y.shape}')

## 4. Leakage-Safe Split

Stratified 50/50: meta-train and meta-test. Prevents leakage
since base models already saw all data during training.

In [ ]:
X_meta_train, X_meta_test, y_meta_train, y_meta_test = (
    train_test_split(X, y, test_size=0.5,
                     random_state=SEED, stratify=y))

print(f'Meta-train: {X_meta_train.shape[0]} samples')
print(f'Meta-test:  {X_meta_test.shape[0]} samples')
print(f'Meta-train dist: {np.bincount(y_meta_train)}')
print(f'Meta-test dist:  {np.bincount(y_meta_test)}')

## 5. Train Meta-Learner

Logistic Regression with multi-class support.

In [ ]:
meta_model = LogisticRegression(
    max_iter=1000, random_state=SEED,
    multi_class='multinomial', solver='lbfgs')
meta_model.fit(X_meta_train, y_meta_train)

train_pred = meta_model.predict(X_meta_train)
train_acc = accuracy_score(y_meta_train, train_pred) * 100
print(f'Meta-learner training accuracy: {train_acc:.2f}%')

## 6. Evaluate

Evaluate ensemble on meta-test and compare with base models.

In [ ]:
# Ensemble predictions
ensemble_pred = meta_model.predict(X_meta_test)
ensemble_proba = meta_model.predict_proba(X_meta_test)

ens_acc = accuracy_score(y_meta_test, ensemble_pred) * 100
ens_prec, ens_rec, ens_f1, _ = precision_recall_fscore_support(
    y_meta_test, ensemble_pred, average='macro')

print('ENSEMBLE RESULTS (meta-test)')
print(f'  Accuracy:  {ens_acc:.2f}%')
print(f'  Precision: {ens_prec*100:.2f}%')
print(f'  Recall:    {ens_rec*100:.2f}%')
print(f'  F1-Score:  {ens_f1*100:.2f}%')

print('\nClassification Report:')
print(classification_report(
    y_meta_test, ensemble_pred, target_names=CLASS_NAMES))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_meta_test, ensemble_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Ensemble Confusion Matrix (meta-test)')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(ENSEMBLE_DIR,
            'ensemble_confusion_matrix.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare ensemble vs base models on same meta-test
rows = []
for i, key in enumerate(MODEL_ORDER):
    start_col = i * NUM_CLASSES
    end_col = start_col + NUM_CLASSES
    m_probs = X_meta_test[:, start_col:end_col]
    m_pred = m_probs.argmax(axis=1)
    acc = accuracy_score(y_meta_test, m_pred) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_meta_test, m_pred, average='macro')
    rows.append({'model': key, 'accuracy': round(acc, 2),
                 'precision': round(prec*100, 2),
                 'recall': round(rec*100, 2),
                 'f1': round(f1*100, 2)})

rows.append({'model': 'ENSEMBLE',
             'accuracy': round(ens_acc, 2),
             'precision': round(ens_prec*100, 2),
             'recall': round(ens_rec*100, 2),
             'f1': round(ens_f1*100, 2)})

comp_df = pd.DataFrame(rows).sort_values(
    'accuracy', ascending=False).reset_index(drop=True)
print('\nCOMPARISON (same meta-test split):')
print(comp_df.to_string(index=False))
comp_df.to_csv(os.path.join(ENSEMBLE_DIR,
               'ensemble_comparison.csv'), index=False)

## 7. ROC Curves (One-vs-Rest)

Multi-class ROC using one-vs-rest strategy.

In [ ]:
y_test_bin = label_binarize(
    y_meta_test, classes=list(range(NUM_CLASSES)))

plt.figure(figsize=(8, 6))
colors = ['#2ecc71', '#e74c3c', '#3498db']

for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(
        y_test_bin[:, i], ensemble_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'{CLASS_NAMES[i]} (AUC={roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0, 1]); plt.ylim([0, 1.02])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Ensemble ROC Curves (One-vs-Rest)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(ENSEMBLE_DIR, 'roc_curves.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Artifacts

Save meta-learner model, config, and metrics.

In [ ]:
# Save meta-learner
meta_path = os.path.join(ENSEMBLE_DIR, 'meta_model.pkl')
with open(meta_path, 'wb') as f:
    pickle.dump(meta_model, f)
print(f'Saved: meta_model.pkl')

# Save ensemble config
config = {
    "model_order": MODEL_ORDER,
    "save_names": SAVE_NAMES,
    "class_names": CLASS_NAMES,
    "meta_learner": "LogisticRegression",
    "feature_dim": 9,
    "ensemble_accuracy": round(ens_acc, 2)
}
cfg_path = os.path.join(ENSEMBLE_DIR,
           'chest_xray_ensemble_config.json')
with open(cfg_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'Saved: chest_xray_ensemble_config.json')

# Save metrics
ens_metrics = {
    "accuracy": round(ens_acc, 2),
    "precision": round(ens_prec * 100, 2),
    "recall": round(ens_rec * 100, 2),
    "f1": round(ens_f1 * 100, 2),
    "meta_train_accuracy": round(train_acc, 2),
    "num_meta_train": int(X_meta_train.shape[0]),
    "num_meta_test": int(X_meta_test.shape[0])
}
met_path = os.path.join(ENSEMBLE_DIR, 'ensemble_metrics.json')
with open(met_path, 'w') as f:
    json.dump(ens_metrics, f, indent=2)
print(f'Saved: ensemble_metrics.json')

In [ ]:
from google.colab import files

print('\nAll artifacts:')
for fn in sorted(os.listdir(ENSEMBLE_DIR)):
    fpath = os.path.join(ENSEMBLE_DIR, fn)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {fn} ({size_kb:.1f} KB)')

files.download(meta_path)
files.download(cfg_path)

## 9. Summary

### Saved Artifacts

| File | Description |
|------|-------------|
| `meta_model.pkl` | Trained LogisticRegression |
| `chest_xray_ensemble_config.json` | Ensemble config |
| `ensemble_metrics.json` | Final metrics |
| `ensemble_comparison.csv` | Base vs ensemble |
| `ensemble_confusion_matrix.png` | Confusion matrix |
| `roc_curves.png` | ROC curves |

### Key Results

The ensemble combines EfficientNet-B0, ResNet-50, and DenseNet-121
through learned probability weighting, typically achieving higher
accuracy than any individual model.

**Integration:** Load `meta_model.pkl` and config in the
NeuraSight backend for production inference.